In [1]:
# ── 1. Imports ──────────────────────────────────────────────
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# ── 2. Load Data ─────────────────────────────────────────────
orders = pd.read_csv('OrderItems.csv')
deliveries = pd.read_csv('Deliveries.csv')

# ── 3. Merge ──────────────────────────────────────────────────
df = orders.merge(deliveries[["OrderID", "LineID", "DeliveryStatus"]],
                  on=["OrderID", "LineID"],
                  how="left")

# ── 4. Create Target ──────────────────────────────────────────
df["is_late"] = (df["DeliveryStatus"] == "Late").astype(int)

# ── 5. Build Features ─────────────────────────────────────────
X = df[["ItemCategory", "Quantity", "UnitPrice", "RequestedDate", "LineAmount"]]
y = df["is_late"]

# Fix dates
X = X.copy()
X["RequestedDate"] = pd.to_datetime(X["RequestedDate"])
X["Month"] = X["RequestedDate"].dt.month
X["DayOfWeek"] = X["RequestedDate"].dt.dayofweek
X["Year"] = X["RequestedDate"].dt.year
X.drop(columns=["RequestedDate"], inplace=True)

# Encode categories
X = pd.get_dummies(X, columns=["ItemCategory"]).astype(int)

# ── 6. Split ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 7. Train ──────────────────────────────────────────────────
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ── 8. Evaluate ───────────────────────────────────────────────
print(classification_report(y_test, model.predict(X_test)))

# ── 9. Feature Importance ─────────────────────────────────────
importances = pd.Series(model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False))

              precision    recall  f1-score   support

           0       0.50      0.43      0.46       437
           1       0.52      0.59      0.55       463

    accuracy                           0.51       900
   macro avg       0.51      0.51      0.51       900
weighted avg       0.51      0.51      0.51       900

LineAmount                  0.234573
Quantity                    0.222047
UnitPrice                   0.209657
Month                       0.123235
DayOfWeek                   0.101406
Year                        0.035396
ItemCategory_Software       0.016822
ItemCategory_Equipment      0.014933
ItemCategory_Hardware       0.014656
ItemCategory_Services       0.014265
ItemCategory_Consumables    0.013010
dtype: float64
